In [1]:
import findspark
import pyspark
from pyspark.sql import SparkSession

In [2]:
findspark.init()
sc = pyspark.SparkContext.getOrCreate()

In [3]:
"""
RDD SOLUTION
"""

'\nRDD SOLUTION\n'

In [11]:
inputRDD = sc.textFile("data/sensors.txt")
inputRDD.collect()

['s1,2016-01-01,20.5',
 's2,2016-01-01,30.1',
 's1,2016-01-02,60.2',
 's2,2016-01-02,20.4',
 's1,2016-01-03,55.5',
 's2,2016-01-03,52.5']

In [12]:
filteredRDD = inputRDD.filter(lambda x: float(x.split(",")[2]) >50)
filteredRDD.collect()

['s1,2016-01-02,60.2', 's1,2016-01-03,55.5', 's2,2016-01-03,52.5']

In [13]:
mappedRDD = filteredRDD.map(lambda x: (x.split(",")[0], 1))
mappedRDD.collect()

[('s1', 1), ('s1', 1), ('s2', 1)]

In [14]:
reducedRDD = mappedRDD.reduceByKey(lambda x, y: x + y)
reducedRDD.collect()

[('s1', 2), ('s2', 1)]

In [15]:
finalRDD = reducedRDD.filter(lambda x: x[1] > 1)
finalRDD.collect()

[('s1', 2)]

In [5]:
"""
SPARKSQL SOLUTION
"""

'\nSPARKSQL SOLUTION\n'

In [6]:
spark = SparkSession.builder.getOrCreate()

In [7]:
df = spark.read.load("data/sensors.txt", format="csv", header=False, inferSchema=True, sep=",")
df.show()

+---+----------+----+
|_c0|       _c1| _c2|
+---+----------+----+
| s1|2016-01-01|20.5|
| s2|2016-01-01|30.1|
| s1|2016-01-02|60.2|
| s2|2016-01-02|20.4|
| s1|2016-01-03|55.5|
| s2|2016-01-03|52.5|
+---+----------+----+



In [8]:
df.createOrReplaceTempView("sensors")

In [16]:
spark.sql("SELECT _c0, COUNT(*) FROM sensors "
          "WHERE _c2 > 50 "
          "GROUP BY _c0 "
          "HAVING COUNT(*) > 1").show()

+---+--------+
|_c0|count(1)|
+---+--------+
| s1|       2|
+---+--------+

